# Next-Day Household Energy Forecasting
### Technical presentation · Capstone Step 6

**Goal:** predict tomorrow's household electricity use and support earlier consumption warnings.

UCI household power data · time-aware validation · explainability and fairness audit


## 1 · Problem formulation

| Element | Definition |
|---|---|
| Primary task | Supervised regression |
| Target | Next-day household energy consumption, kWh |
| Alert event | Actual use > 120% of preceding 30-day mean |
| Primary metric | Mean Absolute Error (MAE) |
| Supporting metrics | RMSE, MAPE, R², warning precision and recall |

The forecast is an **early-visibility estimate**, not an official utility bill calculation.


## 2 · Data provenance and scope

<div style="display:grid;grid-template-columns:1fr 1fr;gap:28px">
<div>

### Source

- UCI Individual Household Electric Power Consumption
- One household in Sceaux, France
- December 2006 to November 2010
- **2,075,259** minute-level rows

</div><div>

### Quality profile

- 7 numeric power and sub-metering variables
- **1.2518%** rows with missing measurements
- No duplicate or absent timestamps detected
- Geographic and household representation is limited

</div></div>

<small>Source: UCI Machine Learning Repository, Individual Household Electric Power Consumption.</small>


## 3 · From minute readings to modeling days

1. Parse and validate timestamps
2. Interpolate only short gaps of **60 minutes or less**
3. Aggregate active power to daily kWh
4. Require at least **95% daily coverage**
5. Create shifted lag, rolling, trend, volatility, and calendar features
6. Exclude rows without sufficient historical context

**Result:** 1,147 modeling rows from 1,417 usable daily records; 42 candidate features reduced to 22.


## 4 · Feature design prevents future leakage

All historical consumption features are shifted before rolling calculations.

```python
daily["energy_lag_1"] = daily["daily_kwh"].shift(1)
daily["energy_roll_mean_7"] = daily["daily_kwh"].shift(1).rolling(7).mean()
daily["energy_ewm_7"] = daily["daily_kwh"].shift(1).ewm(span=7).mean()
```

Feature families: recent energy, rolling statistics, exponentially weighted history, voltage behavior, sub-metering mix, and cyclical calendar terms.

Leakage audit: **all automated checks passed**.


## 5 · Experimental design

| Split | Rows | Date range | Role |
|---|---:|---|---|
| Train | 802 | 2007-01-16 to 2009-05-30 | Fit and time-series cross-validation |
| Validation | 172 | 2009-05-31 to 2010-02-23 | Model selection and tuning |
| Test | 173 | 2010-02-24 to 2010-11-25 | One-time final evaluation |

Compared seasonal naive, persistence, Ridge, Random Forest, and XGBoost. XGBoost was selected by validation MAE **before** opening the test set.


## 6 · XGBoost led validation performance

```python
import matplotlib.pyplot as plt
models = ["Seasonal naive", "Ridge", "Random Forest", "XGBoost"]
mae = [5.6091, 4.2613, 4.1311, 4.0415]
colors = ["#94A3B8", "#38BDF8", "#14B8A6", "#0F766E"]
plt.figure(figsize=(9, 4.2))
bars = plt.barh(models, mae, color=colors)
plt.gca().invert_yaxis(); plt.xlabel("Validation MAE (kWh, lower is better)")
plt.bar_label(bars, fmt="%.2f", padding=4)
plt.xlim(0, 6.2); plt.grid(axis="x", alpha=.2); plt.tight_layout()
plt.show()
```

XGBoost reduced validation MAE by **27.95%** versus the seasonal-naive baseline.


## 7 · Final test: useful forecast, calibrated uncertainty

| Metric | XGBoost | Seasonal naive |
|---|---:|---:|
| MAE | **3.7232 kWh** | 5.7282 kWh |
| RMSE | **5.1040 kWh** | 7.8624 kWh |
| MAPE | **18.73%** | 28.62% |
| R² | **0.5458** | -0.0777 |

- MAE improvement: **35.0%**
- Bootstrap 95% MAE interval: **3.2278–4.2559 kWh**
- 90% prediction-band radius: **±8.677 kWh**
- Observed test coverage: **93.06%**


## 8 · Explainability exposes the warning weakness

Top predictive signals included:

1. Previous-day energy use
2. Seven-day exponentially weighted energy
3. Annual cyclical position
4. Fourteen-day rolling maximum
5. Previous-day voltage variability

The point forecast performs well overall, but high-use days are harder:

- Normal-day MAE: **3.1358 kWh**
- High-day MAE: **7.5540 kWh**
- **21.74% recall:** only 5 of 23 high-use test days detected
- High-day underprediction rate: **91.3%**


## 9 · Ethical and fairness interpretation

The dataset has no gender, race, age, income, socioeconomic status, or multiple-household identifiers. Therefore:

- demographic parity, equalized odds, and disparate impact **cannot be validly computed**;
- season, weekday/weekend, and consumption quartile checks are operational subgroup diagnostics, not demographic fairness claims;
- French single-household results cannot be assumed to transfer to Meralco customers.

Main safeguards: consent-based data collection, data minimization, local calibration, drift monitoring, uncertainty display, and human-configurable alert thresholds.


## 10 · Technical conclusion

### What is ready

- Reproducible next-day forecasting pipeline
- Strong improvement over simple baselines
- Time-aware evaluation, uncertainty estimates, SHAP/LIME/PDP/ICE analysis

### What must come next

- Collect diverse Philippine household data
- Tune tiered Watch / Moderate / High warning logic for recall and alert fatigue
- Validate savings through a prospective pilot
- Re-audit subgroup performance, privacy, and temporal drift before deployment

**Conclusion:** a promising forecasting prototype, not yet a deployment-ready warning system.
